# 1. Train the anomaly model

This notebook builds the reusable PatchCore-style model state. ResNet18 is used as a frozen feature extractor; normal training images provide the memory bank, and normal validation images calibrate the pixel and image thresholds.

The final artifact is written to `output/<category>/anomaly_model.npz`. It contains the memory bank, thresholds, and the preprocessing metadata needed by the evaluation and inference workflows.

## Workflow

1. Load configuration and select the computation device.
2. Split normal `train/good` images into memory-bank and validation sets.
3. Extract and sample normal ResNet18 patches.
4. Calibrate thresholds using only normal validation images.
5. Save the model artifact for later use.

In [1]:
import random
from pathlib import Path

from anomaly_model import (
    ResNetFeatureExtractor,
    build_memory_bank,
    calibrate_thresholds,
    create_neighbor_index,
    image_paths,
    load_config,
    save_artifact,
    set_seed,
    get_device,
)

In [2]:
# Change this one value to train a different category.
TARGET_CATEGORY = 'metal_nut'
config_path = Path('config.yaml')
config = load_config(config_path, category=TARGET_CATEGORY)
set_seed(config['seed'])
device = get_device()
dataset_root = config['dataset_root']
output_dir = config['output_dir']
print(f'Target category: {config["category"]}')
print(f'Dataset: {dataset_root}')
print(f'Output: {output_dir}')
print(f'Device: {device}')

Target category: metal_nut
Dataset: C:\Users\Prashant Suthar\Projects\mvtec_project\src\data\metal_nut
Output: C:\Users\Prashant Suthar\Projects\mvtec_project\src\output\metal_nut
Device: cpu


## Split normal images

Only normal images are used to make the model. The validation subset must remain separate so threshold calibration does not evaluate the same images used to build the memory bank.

In [3]:
normal_images = image_paths(dataset_root / 'train' / 'good')
if len(normal_images) < 2:
    raise RuntimeError('At least two normal training images are required.')

shuffled = list(normal_images)
random.Random(config['seed']).shuffle(shuffled)
validation_count = max(1, int(len(shuffled) * config['validation_ratio']))
validation_images = shuffled[:validation_count]
training_images = shuffled[validation_count:]
if not training_images:
    raise RuntimeError('The validation split consumed all normal images.')
print(f'Memory-bank images: {len(training_images)}')
print(f'Validation images: {len(validation_images)}')

Memory-bank images: 176
Validation images: 44


## Build the memory bank

ResNet18 remains frozen. Each normal image contributes spatial feature patches, and the configured memory-bank limit keeps nearest-neighbor lookup manageable.

In [4]:
model = ResNetFeatureExtractor(config['backbone_name']).to(device)
model.eval()
memory_bank = build_memory_bank(
    model, training_images, config['image_size'], device,
    config['memory_bank_size'], config['seed'],
)
neighbor_index = create_neighbor_index(memory_bank, config['num_neighbors'])
print(f'Memory bank shape: {memory_bank.shape}')

Memory bank shape: (5000, 256)


## Calibrate and save

The pixel threshold controls localization masks. The image threshold controls the final anomalous/normal decision. Both are estimated from normal validation images only.

In [5]:
thresholds = calibrate_thresholds(
    model, neighbor_index, validation_images, config, device
)
artifact_path = output_dir / 'anomaly_model.npz'
save_artifact(artifact_path, memory_bank, thresholds, config)
print(f'Pixel threshold: {thresholds.pixel_threshold:.4f}')
print(f'Image threshold: {thresholds.image_threshold:.4f}')
print(f'Saved model artifact: {artifact_path}')

Pixel threshold: 0.8270
Image threshold: 1.0000
Saved model artifact: C:\Users\Prashant Suthar\Projects\mvtec_project\src\output\metal_nut\anomaly_model.npz


## Notes

Run this notebook again whenever the dataset, backbone settings, or threshold percentiles change. The artifact is category-specific because `config.yaml` selects the dataset category and output directory.